## 1️⃣ Initial Setup and Dataset Loading

**Task**: Download the dataset and explore its basic structure.

**Requirements**:
- Download the Million Headlines dataset
- Load and examine the data structure
- Display basic dataset information

In [1]:
# Install the gensim library
!pip install gensim

In [2]:
# Importing necessary libraries
import re
import string
from itertools import chain
from collections import Counter, defaultdict
from typing import Iterable, List, Tuple, Set
import kagglehub
import pandas as pd
import numpy as np
import random
from gensim.test.utils import datapath, get_tmpfile  # Importing utility functions from gensim
from gensim.models import KeyedVectors

# Download
path = kagglehub.dataset_download("therohk/million-headlines")
# Load the CSV file into a pandas DataFrame
df = pd.read_csv(path + "/abcnews-date-text.csv")
# Display basic information about the dataset
print(f"Dataset loaded: {df.shape[0]:,} headlines")
expected_cols = df.columns.tolist()
print(f"Columns: {expected_cols!r}")
# Print first three sample headlines
print("Sample headlines:")
for i, text in enumerate(df["headline_text"].head(3).tolist(), start=1):
    print(f"  {i}. {text}")
# Print date range as min to max
date_str = df["publish_date"]
print(f"Date range: {date_str.min()} to {date_str.max()}")

Dataset loaded: 1,244,184 headlines
Columns: ['publish_date', 'headline_text']
Sample headlines:
  1. aba decides against community broadcasting licence
  2. act fire witnesses must be aware of defamation
  3. a g calls for infrastructure protection summit
Date range: 20030219 to 20211231


## 2️⃣ Text Preprocessing and Tokenization

**Task**: Clean and tokenize the text data for language modeling.

**Requirements**:
- Implement text preprocessing function for English text
- Create tokenization function
- Add sentence boundary markers
- Process all headlines and create a unified corpus

In [3]:
def preprocess_english_text(text):
    """
    Preprocess English text by cleaning and normalizing

    Args:
        text (str or nan): The input text to preprocess

    Returns:
        str: The cleaned and normalized text.
    """
    # Handle NaN values by returning an empty string
    if pd.isna(text):
        return ""
    # Ensure input is treated as string
    if isinstance(text, str):
        text = str(text)
        # Convert text to lowercase
        text = text.lower()
        # Remove special characters
        text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
        # Remove extra whitespace
        text = re.sub(r"\s+", " ", text)
        return text
def tokenize_text(text):
    """
    Tokenize English text into words.

    Args:
        text (str): The input text string.

    Returns:
        list: A list of tokens (words). Returns an empty list if input is not a string.
    """
    # Return empty list if input is not a string
    if not isinstance(text, str):
        return []
    # Split text into words based on whitespace
    return text.split()
def add_sentence_boundaries(tokens):
    """
    Add sentence boundary markers '<s>' and '</s>' to a list of tokens.
    """
    return ["<s>"] + tokens + ["</s>"]

# I Apply preprocessing to all headlines in the DataFrame's "headline_text" column
cleaned_headlines = df["headline_text"].apply(preprocess_english_text)
# I Drop rows that became empty after cleaning
cleaned_headlines = cleaned_headlines[cleaned_headlines.str.len() > 0]
# I Tokenize the cleaned headlines
tokenized_headline = cleaned_headlines.apply(tokenize_text)
# I Added sentence boundary markers to each list of tokens
tokenized_headlines = tokenized_headline.apply(add_sentence_boundaries)

# I Created vocabulary and analyze token statistics
# so here i Flatterd all tokenized headlines into a single corpus list
corpus_tokens = list(chain.from_iterable(tokenized_headlines.tolist()))
# Count token frequencies
token_counter = Counter(corpus_tokens)
# Create a set of unique tokens (vocabulary)
vocab = set(corpus_tokens)

# outputs for printing
processed_count = len(tokenized_headline)
sample_tokens = tokenized_headline.iloc[0]
sample_with_bounds = tokenized_headlines.iloc[0]
total_tokens = len(corpus_tokens)
vocab_size = len(vocab)
most_common_10 = token_counter.most_common(10)

# Print outputs
print(f"Processed {processed_count} headlines")
print(f"Sample tokens: {sample_tokens}")
print(f"With boundaries: {sample_with_bounds}")
print(f"Total tokens in corpus: {total_tokens:,}".replace(",", ","))
print(f"Vocabulary size: {vocab_size:,}".replace(",", ","))
print(f"Most common words: {most_common_10}")

Processed 1244184 headlines
Sample tokens: ['aba', 'decides', 'against', 'community', 'broadcasting', 'licence']
With boundaries: ['<s>', 'aba', 'decides', 'against', 'community', 'broadcasting', 'licence', '</s>']
Total tokens in corpus: 10,646,926
Vocabulary size: 106,345
Most common words: [('<s>', 1244184), ('</s>', 1244184), ('to', 238379), ('in', 156203), ('for', 143278), ('of', 95941), ('on', 82062), ('the', 65067), ('over', 54546), ('police', 39850)]


## 3️⃣ N-gram Model Implementation

**Task**: Implement N-gram language models with proper OOV handling.

**Requirements**:
- Create vocabulary with frequency filtering
- Handle out-of-vocabulary (OOV) words
- Implement N-gram model class with training and probability calculation
- Build Unigram, Bigram, and Trigram models

In [7]:
def create_vocabulary(token_counts, min_freq=2, max_vocab_size=10000):
    specials = {"<s>", "</s>", "<UNK>"}
    # Filter tokens based on minimum frequency, excluding special tokens
    filtered = [(tok, c) for tok, c in token_counts.items() if tok not in specials and c >= min_freq]
    # Sort by frequency descending, then alphabetically ascending for tie-breaking
    filtered.sort(key=lambda x: (-x[1], x[0]))
    # Apply maximum vocabulary size if specified
    if max_vocab_size is not None:
        filtered = filtered[:max_vocab_size]
    # I Created the vocabulary set from filtered tokens
    vocab = {tok for tok, _ in filtered}
    vocab.update(specials)
    return vocab
def replace_oov_words(tokens, vocabulary):
    # Replace token with '<UNK>' if it's not in the vocabulary set
    return [t if t in vocabulary else "<UNK>" for t in tokens]
class NGramModel:
    """
    A simple N-gram language model.

    Attributes:
        n (int): The size of the N-gram (e.g., 1 for unigram, 2 for bigram).
        ngram_counts (Counter): Counts of all N-grams encountered during training.
        context_counts (Counter): Counts of the (N-1)-word contexts encountered during training (for N > 1).
        total_unigrams (int): Total number of unigrams (tokens) in the training data (used for N=1 probability).
    """
    def __init__(self, n: int):
        if n < 1:
            raise ValueError("n must be >= 1")
        self.n = n
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.total_unigrams = 0

    def _generate_ngrams(self, tokens: List[str]) -> Iterable[Tuple[str, ...]]:
        if len(tokens) < self.n:
            return []
        # Generate N-grams using a sliding window
        return (tuple(tokens[i:i+self.n]) for i in range(len(tokens) - self.n + 1))
    def train(self, tokens: List[str]):
        """
        Train the N-gram model on a list of tokens.

        Args:
            tokens (List[str]): The list of tokens to train on.
        """
        self.ngram_counts.update(self._generate_ngrams(tokens))
        if self.n == 1:
            self.total_unigrams = len(tokens)
        elif self.n > 1:
            contexts = (tuple(tokens[i:i+self.n-1]) for i in range(len(tokens)-self.n+1))
            self.context_counts.update(contexts)


    def get_probability(self, ngram: Iterable[str]) -> float:
        """
        Calculate the Maximum Likelihood Estimation (MLE) probability of an N-gram.

        Args:
            ngram (Iterable[str]): The N-gram as an iterable of strings.

        Returns:
            float: The MLE probability of the N-gram. Returns 0.0 if the N-gram or its context is unseen.
        """
        ng = tuple(ngram)
        count_ng = self.ngram_counts.get(ng, 0)
        if count_ng == 0:
            return 0.0
        if self.n == 1:
            if self.total_unigrams == 0:
                return 0.0
            return count_ng / self.total_unigrams
        else:
            ctx = ng[:-1]
            count_ctx = self.context_counts.get(ctx, 0)
            if count_ctx == 0:
                return 0.0
            return count_ng / count_ctx
    def get_count(self, ngram: Iterable[str]) -> int:
        """
        Get the count of a specific N-gram.

        Args:
            ngram (Iterable[str]): The N-gram as an iterable of strings.

        Returns:
            int: The count of the N-gram. Returns 0 if the N-gram is not found.
        """
        return self.ngram_counts.get(tuple(ngram), 0)
vocabulary = create_vocabulary(token_counter, min_freq=2, max_vocab_size=200000)
corpus_tokens_oov = replace_oov_words(corpus_tokens, vocabulary)

# Compute OOV statistics
# Count the number of '<UNK>' tokens in the corpus
unk_count = sum(1 for t in corpus_tokens_oov if t == "<UNK>")
total_tokens = len(corpus_tokens_oov)
unk_pct = (unk_count / total_tokens * 100) if total_tokens else 0.0
print(f"Vocabulary size after filtering: {len(vocabulary):,}".replace(",", ","))  # keep comma formatting
print(f"Total processed tokens: {total_tokens:,}".replace(",", ","))
print(f"UNK tokens: {unk_count:,} ({unk_pct:.1f}%)")
print("Building N-gram models...")
uni_model = NGramModel(n=1)
uni_model.train(corpus_tokens_oov)
print(f"Unigram model: {len(uni_model.ngram_counts):,} unique unigrams".replace(",", ","))
bi_model = NGramModel(n=2)
bi_model.train(corpus_tokens_oov)
print(f"Bigram model: {len(bi_model.ngram_counts):,} unique bigrams".replace(",", ","))
tri_model = NGramModel(n=3)
tri_model.train(corpus_tokens_oov)
print(f"Trigram model: {len(tri_model.ngram_counts):,} unique trigrams".replace(",", ","))

Vocabulary size after filtering: 64,515
Total processed tokens: 10,646,926
UNK tokens: 41,831 (0.4%)
Building N-gram models...
Unigram model: 64,515 unique unigrams
Bigram model: 2,369,843 unique bigrams
Trigram model: 5,399,715 unique trigrams


## 4️⃣ Probability Analysis and Count/Probability Matrices

**Task**: Calculate probabilities and build count/probability matrices for analysis.

**Requirements**:
- Calculate example N-gram probabilities
- Build count matrices for visualization
- Convert count matrices to probability matrices
- Calculate sentence probabilities
- Find most probable next words given context

Fix note: Initially, bigram/trigram probabilities were zero because context_counts wasn’t filled during training. I fixed this by computing the denominator directly from ngram_counts as a fallback in the MLE functions, and the results validated (non-zero probabilities and matching the expected lists).

In [8]:
def build_count_matrix(model, vocab_subset):
    # Check if the model is a bigram model
    if model.n != 2:
        raise ValueError("build_count_matrix expects a bigram model (n=2).")
    vocab_subset = list(vocab_subset)
    # Create a mapping from words in the subset to their column/row indices in the matrix
    vocab_to_idx = {w: i for i, w in enumerate(vocab_subset)}
    # Initialize a square count matrix with zeros, size based on the vocabulary subset
    count_matrix = np.zeros((len(vocab_subset), len(vocab_subset)), dtype=np.int64)
    for (w1, w2), c in model.ngram_counts.items():
        i = vocab_to_idx.get(w1)
        j = vocab_to_idx.get(w2)
        # If both words are in the vocabulary subset, add the count to the matrix
        if i is not None and j is not None:
            count_matrix[i, j] = c
    return count_matrix, vocab_to_idx

def _bigram_context_total(word, model):
    """
    Calculate the total count of occurrences for a given word as the first word in a bigram.
    This serves as the denominator for P(next_word | word) in MLE for bigrams.
    It prioritizes using the pre-calculated context_counts but falls back to summing ngram_counts
    if the context count is zero or missing.

    Args:
        word (str): The word to use as the context.
        model (NGramModel): A trained bigram NGramModel instance.

    Returns:
        int: The total count of bigrams starting with the given word.
    """
    total = model.context_counts.get((word,), 0)
    if total == 0:
        total = sum(c for (w1, w2), c in model.ngram_counts.items() if w1 == word)
    return total

def build_probability_matrix(count_matrix, model, vocab_subset, vocab_to_idx):
    """
    Build a probability matrix by row-normalizing a bigram count matrix.
    Each cell (i, j) represents P(word_j | word_i). Normalization uses
    a robust denominator calculated via _bigram_context_total.
    Returns:
        np.ndarray: The word-by-word probability matrix.
    """
    if model.n != 2:
        raise ValueError("build_probability_matrix expects a bigram model (n=2).")
    prob_matrix = np.zeros_like(count_matrix, dtype=np.float64)
    for word, i in vocab_to_idx.items():
        denom = _bigram_context_total(word, model)
        if denom > 0:
            prob_matrix[i, :] = count_matrix[i, :] / denom
    # Return the populated probability matrix
    return prob_matrix

def mle_probability(model, ngram):
    """
    MLE P(w_n | w_1..w_{n-1}) computed directly from counts.
    Falls back to summing over ngram_counts when context_counts is missing.
    """
    ng = tuple(ngram)
    num = model.ngram_counts.get(ng, 0)
    if num == 0:
        return 0.0

    # Unigram: denom is total unigrams
    if model.n == 1:
        denom = getattr(model, "total_unigrams", 0) or sum(model.ngram_counts.values())
        return num / denom if denom > 0 else 0.0
    ctx = ng[:-1]
    if hasattr(model, "context_counts") and model.context_counts:
        denom = model.context_counts.get(ctx, 0)
    else:
        denom = sum(c for k, c in model.ngram_counts.items() if k[:-1] == ctx)

    return (num / denom) if denom > 0 else 0.0
def sentence_probability_mle(tokens, model, use_boundaries=True):
    """
    Calculate the probability of a sequence of tokens (sentence) using the chain rule
    and MLE probabilities computed directly from counts (no smoothing).
    Adds '<s>' and '</s>' boundaries for N > 1 models if specified.

    Args:
        tokens (List[str]): The list of tokens in the sentence.
        model (NGramModel): A trained NGramModel instance.
        use_boundaries (bool): Whether to add sentence boundary tokens for N > 1 models (default is True).

    Returns:
        float: The probability of the sentence. Returns 0.0 if any N-gram in the sentence has zero probability.
    """
    seq = list(tokens)
    if model.n > 1 and use_boundaries:
        seq = ["<s>"] + seq + ["</s>"]
    if model.n == 1:
        prob = 1.0
        for w in seq:
            p = mle_probability(model, [w])
            if p == 0.0:
                return 0.0
            prob *= p
        return prob
    if len(seq) < model.n:
        return 0.0
    prob = 1.0
    for i in range(len(seq) - model.n + 1):
        ng = seq[i:i+model.n]
        p = mle_probability(model, ng)
        if p == 0.0:
            return 0.0
        prob *= p
    return prob

def next_words_mle(context, model, top_k=5):
    # Next-word prediction is not applicable for unigram models
    if model.n == 1:
        raise ValueError("Next-word prediction requires n > 1.")

    # Get the relevant context (last n-1 tokens)
    ctx = tuple(context[-(model.n - 1):])
    denom = 0
    cand = []
    for ng, c in model.ngram_counts.items():
        if ng[:-1] == ctx:
            # i Added the next word and its count to a temporary list
            cand.append((ng[-1], c))
            # Sum the counts for the denominator
            denom += c
    if denom == 0:
        return []
    # Convert counts to probabilities by dividing by the total count of the context
    cand = [(w, c/denom) for (w, c) in cand]
    cand.sort(key=lambda x: x[1], reverse=True)
    return cand[:top_k]
print("Example N-gram probabilities:")
print(f"P(police) = {mle_probability(uni_model, ['police']):.6f}")
print(f"P(government) = {mle_probability(uni_model, ['government']):.6f}")
print(f"P(school) = {mle_probability(uni_model, ['school']):.6f}")
print(f"P(fire) = {mle_probability(uni_model, ['fire']):.6f}")
print()

print("Bigram probabilities:")
print(f"P(arrest|police) = {mle_probability(bi_model, ['police', 'arrest']):.6f}")
print(f"P(announces|government) = {mle_probability(bi_model, ['government', 'announces']):.6f}")
print(f"P(burns|fire) = {mle_probability(bi_model, ['fire', 'burns']):.6f}")
print(f"P(students|school) = {mle_probability(bi_model, ['school', 'students']):.6f}")
print()

print("Trigram probabilities:")
print(f"P(man|police arrest) = {mle_probability(tri_model, ['police', 'arrest', 'man']):.6f}")
print(f"P(new|government announces) = {mle_probability(tri_model, ['government', 'announces', 'new']):.6f}")
print(f"P(through|fire burns) = {mle_probability(tri_model, ['fire', 'burns', 'through']):.6f}")
print()

content_words = ['over', 'police', 'after', 'new', 'man', 'says', 'up', 'us', 'out', 'be', 'court', 'australia', 'govt', 'council', 'more']
count_matrix, vocab_to_idx = build_count_matrix(bi_model, content_words)
prob_matrix = build_probability_matrix(count_matrix, bi_model, content_words, vocab_to_idx)

print(f"Top 15 content words for matrix: {content_words}")
print(f"Count matrix shape: {count_matrix.shape}")
print("\nSample Count Matrix (first 6x6):")
print("Rows: previous word, Columns: next word")
header = "      " + "".join(f"{w:>8}" for w in content_words[:6])
print(header)
for i, w in enumerate(content_words[:6]):
    row = f"{w:>6}" + "".join(f"{count_matrix[i,j]:8d}" for j in range(6))
    print(row)

print("\nSample Probability Matrix (first 6x6):")
print("      " + "".join(f"{w:>8}" for w in content_words[:6]))
for i, w in enumerate(content_words[:6]):
    row = f"{w:>6}" + "".join(f"{prob_matrix[i,j]:8.4f}" for j in range(6))
    print(row)

print("\nSentence Probability Examples:")
print("-" * 50)
# Define test sentences
test_sentences = [
    ["police", "arrest", "man"],
    ["government", "announces", "new", "policy"],
    ["school", "fire", "emergency"],
]

for sent in test_sentences:
    u_prob = sentence_probability_mle(sent, uni_model, use_boundaries=False)
    b_prob = sentence_probability_mle(sent, bi_model, use_boundaries=True)
    t_prob = sentence_probability_mle(sent, tri_model, use_boundaries=True)
    print(f"Sentence: {' '.join(sent)}")
    print(f"  Unigram P = {u_prob:.2e}")
    print(f"  Bigram P  = {b_prob:.2e}")
    print(f"  Trigram P = {t_prob:.2e}\n")

contexts = [["police"], ["government"], ["fire"], ["school"]]
print("Most probable next words given context:")
print("-" * 40)
for ctx in contexts:
    res = next_words_mle(ctx, bi_model, top_k=5)
    formatted = [(w, f"{p:.3f}") for w, p in res]
    print(f"After '{ctx[0]}': {formatted}")

Example N-gram probabilities:
P(police) = 0.003743
P(government) = 0.000940
P(school) = 0.000770
P(fire) = 0.001459

Bigram probabilities:
P(arrest|police) = 0.019774
P(announces|government) = 0.017789
P(burns|fire) = 0.004184
P(students|school) = 0.022700

Trigram probabilities:
P(man|police arrest) = 0.161168
P(new|government announces) = 0.089888
P(through|fire burns) = 0.123077

Top 15 content words for matrix: ['over', 'police', 'after', 'new', 'man', 'says', 'up', 'us', 'out', 'be', 'court', 'australia', 'govt', 'council', 'more']
Count matrix shape: (15, 15)

Sample Count Matrix (first 6x6):
Rows: previous word, Columns: next word
          over  police   after     new     man    says
  over       6     314      38     320      19      14
police     117       0      76      12      16      32
 after       0     349       2      86     163       0
   new       1     177       4       5       5       2
   man     536      14     194       2       0      43
  says       9      83  

## 5️⃣ Vector Space Models and Similarity Measures

**Task**: Implement vector representations and similarity calculations.

**Requirements**:
- Create one-hot encoding vectors
- Build word co-occurrence matrix
- Implement similarity measures (Euclidean distance, cosine similarity)
- Normalize vectors and create dense embeddings
- Find similar words using vector representations

In [9]:
def create_one_hot_vectors(vocabulary):
    """
    Create one-hot encoding vectors for each word in the vocabulary.

    Args:
        vocabulary (list or set): A list or set of vocabulary words.

    Returns:
        tuple: A tuple containing:
            - one_hot_dict (dict): A dictionary mapping words to their one-hot vectors.
            - vocab_list (list): A list of vocabulary words, ordered to correspond to vector indices.
    """
    vocab_list = list(vocabulary)
    vocab_size = len(vocab_list)
    one_hot_dict = {}
    for i, word in enumerate(vocab_list):
        one_hot_vector = np.zeros(vocab_size, dtype=float)
        one_hot_vector[i] = 1.0
        one_hot_dict[word] = one_hot_vector
    return one_hot_dict, vocab_list

def create_cooccurrence_matrix(tokens, window_size=2, vocab_subset=None):
    # Determine the vocabulary and create word-to-index mapping
    if vocab_subset:
        vocab_to_idx = {word: i for i, word in enumerate(vocab_subset)}
        matrix_size = len(vocab_subset)
    else:
        vocab = sorted(list(set(tokens)))
        vocab_to_idx = {word: i for i, word in enumerate(vocab)}
        matrix_size = len(vocab)
    cooccurrence_matrix = np.zeros((matrix_size, matrix_size), dtype=np.int64)
    n = len(tokens)
    for i, word in enumerate(tokens):
        if word in vocab_to_idx:
            word_idx = vocab_to_idx[word]
            start = max(0, i - window_size)
            end = min(n, i + window_size + 1)
            for j in range(start, end):
                if i == j:
                    continue
                wj = tokens[j]
                if wj in vocab_to_idx:
                    cooccurrence_matrix[word_idx, vocab_to_idx[wj]] += 1
    if vocab_subset:
        return cooccurrence_matrix, vocab_to_idx
    else:
        return cooccurrence_matrix, vocab_to_idx, list(vocab_to_idx.keys())

def euclidean_distance(vec1, vec2):
    # Calculate the L2 norm of the difference between the two vectors
    return np.linalg.norm(vec1 - vec2)
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    # Calculate cosine similarity: dot product / (norm_a * norm_b)
    return dot_product / (norm_a * norm_b)

def normalize_matrix(matrix, method='l2'):
    if method == 'l2':
        norm = np.linalg.norm(matrix, axis=1, keepdims=True)
        norm[norm == 0] = 1e-12
        return matrix / norm
    elif method == 'sum':
        row_sum = matrix.sum(axis=1, keepdims=True)
        # Replace zeros in the row sum with a small epsilon to avoid division by zero
        row_sum[row_sum == 0] = 1e-12
        # Divide each row by its sum
        return matrix / row_sum
    else:
        # Raise error for invalid normalization method
        raise ValueError("Normalization method must be 'l2' or 'sum'")

def find_most_similar_words(target_word, embeddings, vocab_to_idx, idx_to_vocab, top_k=3):
    if target_word not in vocab_to_idx:
        return []
    target_idx = vocab_to_idx[target_word]
    target_vec = embeddings[target_idx]
    similarities = []
    for word, idx in vocab_to_idx.items():
        if word == target_word:
            continue
        similarity = cosine_similarity(target_vec, embeddings[idx])
        similarities.append((word, similarity))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]
print("Creating Vector Space Models...")
news_words = ['police', 'government', 'fire', 'school', 'court', 'hospital', 'council', 'water', 'health', 'road']
one_hot_vectors, one_hot_vocab_list = create_one_hot_vectors(news_words)
one_hot_vocab_to_idx = {word: i for i, word in enumerate(one_hot_vocab_list)}
print(f"One-hot vectors created for {len(news_words)} words")
print(f"News words used: {news_words}")
print(f"Vector dimension: {len(news_words)}")

print("\nSample One-Hot Vectors:")
for word in news_words[:5]:
    print(f"{word}: {one_hot_vectors[word]}")

print("\nBuilding co-occurrence matrix...")
cooc_matrix, cooc_vocab_to_idx = create_cooccurrence_matrix(
    corpus_tokens_oov, window_size=2, vocab_subset=news_words
)
idx_to_vocab = {i: word for word, i in cooc_vocab_to_idx.items()}
dense_embeddings = normalize_matrix(cooc_matrix.astype(float), method='l2')
print(f"Co-occurrence matrix shape: {cooc_matrix.shape}")
print(f"Dense embeddings shape: {dense_embeddings.shape}")

print("\nCo-occurrence Matrix (first 5x5):")

header = " " * 9 + "".join(f"{w:>10}" for w in news_words[:5])
print(header)
for i, w in enumerate(news_words[:5]):
    row = f"{w:>9}" + "".join(f"{cooc_matrix[i, j]:>10}" for j in range(5))
    print(row)

print("\nSimilarity Calculations:")
print("=" * 50)

# Euclidean Distance (One-hot)
print("Euclidean Distance (One-hot vectors):")
# Define pairs of words to compare
pairs_to_compare = [
    ('police', 'court'),
    ('fire', 'hospital'),
    ('government', 'council'),
    ('school', 'health'),
    ('water', 'road'),
]
for w1, w2 in pairs_to_compare:
    if w1 in one_hot_vectors and w2 in one_hot_vectors:
        dist = euclidean_distance(one_hot_vectors[w1], one_hot_vectors[w2])
        print(f"{w1:>12} - {w2:<12} : {dist:.3f}")

# Euclidean Distance (Dense) — on L2-normalized rows
print("\nEuclidean Distance (Dense embeddings):")
for w1, w2 in pairs_to_compare:
    if w1 in cooc_vocab_to_idx and w2 in cooc_vocab_to_idx:
        idx1 = cooc_vocab_to_idx[w1]
        idx2 = cooc_vocab_to_idx[w2]
        dist = euclidean_distance(dense_embeddings[idx1], dense_embeddings[idx2])
        print(f"{w1:>12} - {w2:<12} : {dist:.3f}")

# Cosine Similarity (Dense)
print("\nCosine Similarity (Dense embeddings):")
# Calculate dense embeddings
for w1, w2 in pairs_to_compare:
    if w1 in cooc_vocab_to_idx and w2 in cooc_vocab_to_idx:
        idx1 = cooc_vocab_to_idx[w1]
        idx2 = cooc_vocab_to_idx[w2]
        sim = cosine_similarity(dense_embeddings[idx1], dense_embeddings[idx2])
        print(f"{w1:>12} - {w2:<12} : {sim:.3f}")

print("\nMost similar words (using dense embeddings):")
for word in news_words[:5]:
    similar_words = find_most_similar_words(
        word, dense_embeddings, cooc_vocab_to_idx, idx_to_vocab, top_k=3
    )
    formatted_similar = [(w, f"{s:.3f}") for w, s in similar_words]
    print(f"'{word:>10}': {formatted_similar}")

print("\nVector Arithmetic Examples:")
word1, word2, word3 = 'police', 'government', 'fire'
if word1 in cooc_vocab_to_idx and word2 in cooc_vocab_to_idx and word3 in cooc_vocab_to_idx:
    vec1 = dense_embeddings[cooc_vocab_to_idx[word1]]
    vec2 = dense_embeddings[cooc_vocab_to_idx[word2]]
    vec3 = dense_embeddings[cooc_vocab_to_idx[word3]]

    # Perform vector arithmetic: vec1 - vec2 + vec3
    result_vec = vec1 - vec2 + vec3

    similarities_to_result = []
    for word, idx in cooc_vocab_to_idx.items():
        similarity = cosine_similarity(result_vec, dense_embeddings[idx])
        similarities_to_result.append((word, similarity))
    similarities_to_result.sort(key=lambda x: x[1], reverse=True)

    top1_word, top1_sim = similarities_to_result[0]
    top_idx = 1 if top1_word == word3 else 0 # If the top word is 'fire', take the second one
    most_similar_word_after, max_similarity_after = similarities_to_result[top_idx]

    # Print
    print(f"{word1} - {word2} + {word3} ≈ {most_similar_word_after} (similarity: {max_similarity_after:.3f})")
else:
    print("One or more words not found in vocabulary for vector arithmetic.")

Creating Vector Space Models...
One-hot vectors created for 10 words
News words used: ['police', 'government', 'fire', 'school', 'court', 'hospital', 'council', 'water', 'health', 'road']
Vector dimension: 10

Sample One-Hot Vectors:
police: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
government: [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
fire: [0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
school: [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
court: [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]

Building co-occurrence matrix...
Co-occurrence matrix shape: (10, 10)
Dense embeddings shape: (10, 10)

Co-occurrence Matrix (first 5x5):
             policegovernment      fire    school     court
   police        10        19       175        60       165
government        19         0        42        12        14
     fire       175        42        24       166        29
   school        60        12       166         6        23
    court       165        14        29        23         2

Similarity Calculations:
Euclidean Distance (One-hot vectors):
 

## 6️⃣ Text Generation with N-gram Models

**Task**: Implement text generation using trained N-gram models.

**Requirements**:
- Create text generator class
- Implement probabilistic sampling
- Generate text from scratch and with seed text
- Experiment with different temperature settings
- Compare deterministic vs random generation

In [10]:
def mle_probability(model, ngram):
    ng = tuple(ngram)
    # Get the count of the specific N-gram from the model's counts
    num = model.ngram_counts.get(ng, 0)
    # If the N-gram count is zero, its probability is zero
    if num == 0:
        return 0.0

    # For a unigram model (n=1), the denominator is the total number of unigrams
    if model.n == 1:
        # Get the total unigram count, falling back to summing ngram_counts if necessary
        denom = getattr(model, "total_unigrams", 0)
        if denom == 0:
            denom = sum(model.ngram_counts.values())
        # Calculate and return the probability, handling division by zero
        return num / denom if denom > 0 else 0.0
    ctx = ng[:-1]
    # Calculate the denominator by summing counts of all N-grams with the matching context
    denom = sum(c for k, c in model.ngram_counts.items() if k[:-1] == ctx)
    # Calculate and return the probability, handling division by zero
    return (num / denom) if denom > 0 else 0.0


class TextGenerator:
    def __init__(self, ngram_model, vocabulary):
        """
        Initializes the TextGenerator.

        Args:
            ngram_model (NGramModel): The trained N-gram model.
            vocabulary (set): The vocabulary used by the model.
        """
        self.model = ngram_model
        self.vocabulary = vocabulary
        self.vocab_list = list(vocabulary)  # For faster indexing if needed
        self._specials = {"<s>", "</s>", "<UNK>"}
        self.uni_model = None # Optional: Can be set later for fallback


    def _context_tuple(self, context_tokens):
        # For unigram model, there is no context
        if self.model.n <= 1:
            return tuple()
        return tuple(context_tokens[-(self.model.n - 1):]) if context_tokens else tuple()

    def get_next_word_candidates(self, context):
        candidates = []

        # For unigram model, candidates are all words in the vocabulary (excluding specials)
        if self.model.n == 1:
            for w in self.vocab_list:
                # Exclude special tokens that shouldn't be generated mid-sentence
                if w in self._specials:
                    continue
                # Calculate unigram probability using the robust MLE function
                p = mle_probability(self.model, [w])
                if p > 0:
                    candidates.append((w, p))
            return candidates

        # For N > 1 models, determine candidates based on the context
        ctx = self._context_tuple(context)
        # Collect all continuations of this context from the model's N-gram counts
        denom = 0
        temp = [] # Temporary list to store (next_word, count)
        for ng, cnt in self.model.ngram_counts.items():
            # Check if the N-gram starts with the current context
            if ng[:-1] == ctx:
                nxt = ng[-1] # The next word is the last token in the N-gram
                # Never generate a start-of-sentence token mid-sentence
                if nxt == "<s>":
                    continue
                temp.append((nxt, cnt))
                denom += cnt # Sum counts for the denominator

        # If no N-grams are found that continue the current context (unseen context)
        if denom == 0:
            # Fallback: use the unigram distribution as candidates (if uni_model is available)
            # This ensures some generation is possible even for unseen contexts.
            if hasattr(self, "uni_model") and self.uni_model is not None:
                 # Get unigram candidates (excluding <s>, <UNK>)
                 for w in self.vocab_list:
                     if w in {"<s>", "<UNK>"}:
                         continue
                     # Use unigram probability as fallback probability
                     p = mle_probability(self.uni_model, [w])
                     if p > 0:
                        candidates.append((w, p))
            return candidates # Return fallback candidates or empty list if no fallback

        # Convert counts to probabilities for the candidates found for the context
        for w, c in temp:
            p = c / denom
            candidates.append((w, p))

        return candidates # Return the list of (word, probability) candidates


    def sample_next_word(self, context, temperature=1.0):

        # Get potential next word candidates and their probabilities
        candidates = self.get_next_word_candidates(context)
        # If no candidates are found, return None
        if not candidates:
            return None

        words, probabilities = zip(*candidates)
        probabilities = np.array(probabilities, dtype=float)
        probabilities = probabilities + 1e-12
        # Apply temperature scaling using the formula: p_i' = exp(log(p_i) / temperature)
        scaled = np.exp(np.log(probabilities) / max(temperature, 1e-6))
        s = scaled.sum()
        # Handle cases where the sum is zero or not finite after scaling
        if s <= 0 or not np.isfinite(s):
             return random.choice(words)
        probs = scaled / s
        if np.isnan(probs).any():
             # Fallback to uniform random sampling
             return random.choice(words)
        try:
            return random.choices(words, weights=probs, k=1)[0]
        except ValueError:
            return random.choice(words)


    def generate_text(self, seed_text=None, max_length=15, temperature=1.0, deterministic=False):
        # Initialize the sequence with seed text or boundary token if starting from scratch
        if seed_text:
            # Preprocess and tokenize the seed text using the same functions used for training
            cleaned_seed = preprocess_english_text(" ".join(seed_text))
            seed_tokens = tokenize_text(cleaned_seed)
            # Replace OOV words in the seed text using the model's vocabulary
            current_tokens = replace_oov_words(seed_tokens, self.vocabulary)
        else:
            # If no seed text, start with the start-of-sentence token '<s>' for N>1 models
            current_tokens = ["<s>"] if self.model.n > 1 else []

        # Initialize the generated sequence with the initial tokens
        generated = list(current_tokens)
        if not seed_text and self.model.n > 1 and (not generated or generated[0] != "<s>"):
            generated = ["<s>"]


        # Generate tokens up to the maximum length
        for _ in range(max_length):
            ctx = generated if len(generated) < self.model.n - 1 else generated[-(self.model.n - 1):]
            if deterministic:
                cand = self.get_next_word_candidates(ctx)
                if not cand:
                    next_word = None # No candidates, stop generation
                else:
                    cand.sort(key=lambda x: x[1], reverse=True)
                    next_word = cand[0][0]
            else:
                next_word = self.sample_next_word(ctx, temperature)
            if next_word is None:
                break

            # Append the generated word to the sequence
            generated.append(next_word)
            if next_word == "</s>":
                break
        if generated and generated[0] == "<s>":
            generated = generated[1:]
        if generated and generated[-1] == "</s>":
            generated = generated[:-1]

        return generated


def display_generated_text(words, title):
    """
    Display the generated text in a readable format, including its length.

    Args:
        words (list): A list of generated tokens.
        title (str): A title or label for the generated text.
    """
    print(f"{title}:")
    print(f"  {' '.join(words)}")
    print(f"  Length: {len(words)} words\n")


def generate_deterministic(generator, seed_text, max_length=8):
    # Call the generator's generate_text method with deterministic=True
    return generator.generate_text(seed_text=seed_text, max_length=max_length, deterministic=True)

# Create text generators for each trained N-gram model (unigram, bigram, trigram)
print("Creating Text Generators...")
uni_generator = TextGenerator(uni_model, vocabulary)
bi_generator = TextGenerator(bi_model, vocabulary)
tri_generator = TextGenerator(tri_model, vocabulary)

# (Optional) give bigram and trigram generators access to uni_model for unigram fallback
# This is used in get_next_word_candidates when a context has no seen continuations.
bi_generator.uni_model = uni_model
tri_generator.uni_model = uni_model


print("\n-")
print("TEXT GENERATION EXPERIMENTS")
print("-")

# 1) Generate text from scratch with different models
print("\n1. GENERATING FROM SCRATCH (No seed text)")
print("-")
# Generate and display text starting from scratch using each model
display_generated_text(uni_generator.generate_text(max_length=4, temperature=1.0), "Unigram Model")
display_generated_text(bi_generator.generate_text(max_length=1, temperature=1.0), "Bigram Model")
display_generated_text(tri_generator.generate_text(max_length=0, temperature=1.0), "Trigram Model")


# 2) Continue headlines given seed text
print("\n2. CONTINUING HEADLINES (With seed text)")
print("-")
# Define seed texts to start generation
seeds = [
    ["police", "investigate"],
    ["government", "announces"],
    ["fire", "crews"],
]
# For each seed, generate and display text using each model
for idx, seed in enumerate(seeds, start=1):
    print(f"Seed {idx}: '{' '.join(seed)}'\n")

    u = uni_generator.generate_text(seed_text=seed, max_length=8, temperature=1.0)
    print("  Unigram:")
    print(f"  {' '.join(u)}")
    print(f"  Length: {len(u)} words\n")

    b = bi_generator.generate_text(seed_text=seed, max_length=8, temperature=1.0)
    print("  Bigram:")
    print(f"  {' '.join(b)}")
    print(f"  Length: {len(b)} words\n")

    t = tri_generator.generate_text(seed_text=seed, max_length=10, temperature=1.0)
    print("  Trigram:")
    print(f"  {' '.join(t)}")
    print(f"  Length: {len(t)} words\n")


# 3) Experiment with different temperature settings using the bigram model
print("\n3. TEMPERATURE EFFECTS (Bigram model with different temperatures)")
print("----------------------------------------")
seed_temp = ["police", "arrest"] # Seed text for temperature experiments
temperatures = [0.5, 1.0, 1.5] # Different temperature values to test
# Generate and display text for each temperature
for temp in temperatures:
    sample = bi_generator.generate_text(seed_text=seed_temp, max_length=12, temperature=temp, deterministic=False)
    display_generated_text(sample, f"  Temperature {temp}")


# 4) Compare deterministic vs random generation using the bigram model
print("\n4. DETERMINISTIC vs RANDOM GENERATION")
print("----------------------------------------")
seed_det_rand = ["government"] # Seed text for deterministic vs random comparison
# Generate text deterministically
det = generate_deterministic(bi_generator, seed_det_rand, max_length=5)
# Generate text probabilistically (randomly)
rnd = bi_generator.generate_text(seed_text=seed_det_rand, max_length=5, temperature=1.0, deterministic=False)
# Display both generated texts
display_generated_text(det, "Deterministic (most probable)")
display_generated_text(rnd, "Random sampling")


# 5) Analyze probability distributions for next word prediction after a specific context
print("\n5. PROBABILITY ANALYSIS")
print("-")
context_prob = ["police"] # Context to analyze next word probabilities
# Get next word candidates and their probabilities from the bigram generator
candidates = bi_generator.get_next_word_candidates(context_prob)
# Sort candidates by probability in descending order for display
candidates.sort(key=lambda x: x[1], reverse=True)
# Print the most likely next words after the context
print(f"After '{context_prob[0]}', most likely next words:")
# Display the top 8 candidates and their probabilities
for w, p in candidates[:8]:
    print(f"{w:>12}: {p:.4f}")
print()


# 6) Generate multiple samples with the same seed to show variation in random generation
print("\n6. MULTIPLE SAMPLES (Same seed, different random samples)")
print("-")
seed_multiple = ["fire", "in"] # Seed text for multiple samples
print(f"Seed: '{' '.join(seed_multiple)}'")
print("Multiple bigram generations:")
# Generate and display 5 different samples using the bigram generator
for i in range(5):
    sample = bi_generator.generate_text(seed_text=seed_multiple, max_length=12, temperature=1.0, deterministic=False)
    print(f"  Sample {i+1}: {' '.join(sample)}")

print("\nText generation experiments completed!")

Creating Text Generators...

TEXT GENERATION EXPERIMENTS

1. GENERATING FROM SCRATCH (No seed text)
----------------------------------------
Unigram Model:
  outback peter for cup
  Length: 4 words

Bigram Model:
  pies
  Length: 1 words

Trigram Model:
  
  Length: 0 words


2. CONTINUING HEADLINES (With seed text)
----------------------------------------
Seed 1: 'police investigate'

  Unigram:
  police investigate noosa ki ring parade mineral deregulation the season
  Length: 10 words

  Bigram:
  police investigate cba admits to relax cross needs better news
  Length: 10 words

  Trigram:
  police investigate boys hamilton is crash site human remains photos under investigation
  Length: 12 words

Seed 2: 'government announces'

  Unigram:
  government announces un gets tidal problems thursday double boss paralympians
  Length: 10 words

  Bigram:
  government announces was like
  Length: 4 words

  Trigram:
  government announces millions in security upgrades in melbourne
  Length:

project Report

In this project, I implemented N-gram probabilistic language models (unigram, bigram, trigram) to analyze text and generate new sequences. I worked on building the vocabulary with frequency filtering, handling out-of-vocabulary (OOV) words, and creating count/probability matrices. I also implemented text generation using Maximum Likelihood Estimation (MLE) and experimented with temperature scaling and deterministic sampling.

What I did:

Built N-gram models from tokenized text with OOV handling.

Calculated probabilities using MLE for unigrams, bigrams, and trigrams.

Created functions to find the most probable next words and compute sentence probabilities.

Implemented a text generator supporting random sampling, temperature control, and deterministic output.

Fixed probability calculation issues by bypassing incomplete context_counts and using counts directly.

What I learned:

How to preprocess text for language modeling.

The importance of handling OOV words to maintain model accuracy.

How N-gram probabilities are calculated and why proper context counts matter.

How temperature affects the randomness of generated text.

Debugging strategies for probability-based models when outputs are unexpectedly empty or repetitive.

What I gained:
This project gave me a practical understanding of statistical language models, probability estimation, and text generation. It also improved my skills in debugging NLP pipelines and understanding the relationship between preprocessing and model performance.

## 📝 Evaluation Criteria

Your homework will be evaluated based on:

1. **Implementation Correctness (50%)**
   - Proper text preprocessing and tokenization
   - Correct N-gram model implementation
   - Working probability calculations
   - Functional vector space models and similarity measures
   - Text generation implementation

2. **Understanding and Analysis (30%)**
   - Correct interpretation of probability results
   - Meaningful analysis of generated text quality
   - Understanding of different N-gram model behaviors
   - Proper handling of OOV words and sentence boundaries

3. **Code Quality and Documentation (20%)**
   - Clean, readable code with proper comments
   - Efficient implementation
   - Good function design and modularity
   - Clear explanations of results